## AY25-26 Term 2 CS421 Group Project

### Data and Task

In this project, you will be working with data extracted from a recommender systems type dataset: you are provided with a large set of interactions between subjects (e.g., online shopping users, or social network users)  and items (e.g., shopping items, books, songs, or movies). Whenever a user "interacts" with an item, he/she has some experience the item (e.g., purchase the items, read the book, listen the songs, or watch the movie) and gives a mark or "rating" between 0 and 5 stars (5 stars indicating the best experience, and 1 or 0 stars indicating that the user didn't like the experience at all). 

In this exercise, we will **not** be performing the recommendation task per se. Instead, we will identify *anomalous users*. In the dataset that you are provided with, some of the data was corrupted. Whilst most of the data comes from real life user-item interactions from a real-world application, some "users" are anomalous: they were generated by me according to some undisclosed procedure to simulate exceptional users with unexpected interactions. Most users have normal, expected interactions in real applications, so the anomalous users only account for a very small proportion of the users. Thus, identifying those anomalous users are important to the application, but it can be very challenging.

You are provided with two data frames: the first one ("X") contains the interactions provided to you, and the second one ("y") contains the labels for the users. There is a set of 1,000 unique items the users interact with, ranging from item ID of 0 to item ID 999, which is fixed throughout the competition. Note that it is not necessarily that all items appear in one set of data samples given during the competition, meaning that some of the items may not be rated by any user in a user set within a specific period of time.

As you can see, the three columns in "ratings" correspond to the user ID, the item ID and the rating. Thus, each row of "ratings" contains a single interaction. For instance, if the row "142, 152, 5" is present, this means that the user with ID 142 has given the movie 152 the rating 5 stars.

The dataframe "labels" has two columns. In the first column we have the user ids, whilst the second column contains the labels. A label of 1 indicates that the user is 'anomalous', whilst a label of 0 denotes a natural, normal user (coming from real life interactions). 

For instance, if the labels matrix contains the line "142, 1", it means that all of the ratings given by the user with id 142 is an anomalous user. This means all lines in the dataframe "ratings" which start with the userID 142 correspond to fake interactions. 

### Evaluation

Your task is to be able to classify unseen instances as either anomalies or non anomalies. 

There are **far more** normal users than anomalies in the dataset, which makes this a very heavily **unbalanced dataset**. Thus, accuracy will not be a good measure of performance, since simply predicting that every user is normal will give good accuracy. Thus, we need to use some other evaluation metrics (see slides and lecture note from week 3). 

Suitable **EVALUATION METRICS** include: **AUC** (AREA UNDER CURVE), **PRECISION**, **RECALL**, and **F1 score**. The **main metric** will be the **AREA UNDER CURVE**, and it will by default be used to rank teams. This means your programs should return an **anomaly score** for each user (the higher the score, the more likely the model think the sample is anomalous).  

Before Week 9, each team is required to develop a model using the data `training_data_with_labels.npz` that contains labeled data. You can leverage the data in whatever the way you believe is right. Thereafter every week, we will evaluate the performance of each team's model on an *unseen test set* I will provide, in terms of AUC, and rank the teams by **AUC**. We will release the class labels for the test dataset after we finalize the ranking for each week. Each team then can utilize this newly released class labels (together with the previously released labeled data) to develop a better model.

The difficulty implied by **the generation procedure of the anomalies WILL CHANGE as the project evolves: depending on how well the teams are doing, I will generate easier or harder anomalies**.

The **first round of competition** will take place after recess (week 9): this means that I will **release the first test set on the Thursday of week 9**, and you must submit at least one valid set of anomaly scores to **Codabench** before the **Wednesday of week 10 at 11:59 PM**. Your submission will be **a .npz file** (check the exemplar file attached and the code below that generates the .npz file for detail). We will then look at the results together during Thursday's class. Each group will be limited to no more than **THREE submissions per week**. The best result is taken to rank in the leaderboard.  

We will check everyone's performance in this way every week (once on  week 10, once on week 11, and once on week 12). 

Whilst performance (expressed in terms of AUC and your ranking compared to other teams) at **each of the check points** (weeks 9 to 12 inclusive) is an **important component** of your **final grade**, the **project report** and the detail of the various methods you will have tried will **also** be very **important**. Ideally, to get perfect marks (A+) for this component, you should try at least **two supervised methods** and **one unsupervised methods**, as well as be ranked among the **best teams (top 3)** in terms of performance. 

For the project report, I will be especially interested in your analysis and reasoning. High marks will be given to any team that is able to qualitatively and quantitatively analyze the performance of different versions of your models, delivering convincing empirical justification about why the models behave in a particular way. Exemplar qualitative analyses include visualization of feature representations learned in hidden layers, distribution of anomaly scores for the two classes, reasoning over typical failure and correct cases, etc. In terms of quantitative analysis, in addition to general AUC performance of your overall models, other useful quantitative results may include ablation study (i.e., how each component of your model contributes to the overall performance), hyperparameter sensitivity analysis, analysis of computational cost, etc.


The performance part of the grading will be based the leaderboard ranking of your group averaged over the three weeks (50% on the first two rounds of ranking, and 50% on the last round of ranking).

**Leaderboard Instructions**

Create an account on Codabench using your **school email address**. Then, click on the competition link and register for the competition. Once you complete the registration, you will be automatically enrolled in the competition.

**Only one registration per group** is needed; more registration will not be approved, as we rely on the registration to strictly enforce the THREE submissions per week.

**Submision Format**
1. Save your predictions as a .npz file
   The key **must** be named `predictions`, otherwise the scoring program will not be able to read your submission.
   
2. Zip the file
eg. "submission.npz"  →  "submission.zip"

3. Upload "submission.zip" to Codabench

Competition link : https://www.codabench.org/competitions/14096/

**There's limit on the number of submissions allowed per day and per competition phase, so plan you submissions carefully.

**Phases**
1. **Trial Phase.** From now until Wednesday 11.59pm Week 9. Training data is `training_data_with_labels.npz`. Toy test data: `subset_training_batch.npz`, on which you can make prediction and submit the prediction scores. It is for you to get familiar with Codabench.

2. **Competition Phases.** Each week we release a new test set at Codabench on Thursday (first release will be at week 9).
- Each group must submit at least one valid set of anomaly scores to Codabench before the Wednesday of subsequent week at 11:59 PM. Your group's ranking will be zero if otherwise.
- AUC performance of your submitted anomaly scores will be used for leaderboard ranking.
- The class labels for the test data will be released on `eLearn -> CS421 -> Content -> Group Project` for better training the model for next round
- Up to **THREE** submissions per week per group.

In [2]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

# data=np.load("training_batch_with_labels.npz")
data=np.load("/kaggle/input/datasets/victoriaquek/pml-data/training_batch_with_labels.npz")

In [3]:
X=data["X"]
y=data["y"]

print("# of interactions:", X.shape[0])
print("# of anomalous and normal users:", np.count_nonzero(y==1), np.count_nonzero(y==0))


XX=pd.DataFrame(X)
yy=pd.DataFrame(y)
XX.rename(columns={0:"user",1:"item",2:"rating"},inplace=True)
print("# of items:", XX['item'].unique().shape[0])

# of interactions: 177346
# of anomalous and normal users: 100 1000
# of items: 993


In [4]:
XX.head(100)

,user,item,rating
0,304,0,3
1,304,1,3
2,304,14,3
3,304,17,4
4,304,19,4
...,...,...,...
95,304,824,3
96,304,826,3
97,304,828,4
98,304,847,3


In [5]:
yy.rename(columns={0:"user",1:"label"},inplace=True)

In [6]:
yy.head(10)

,user,label
0,100,0
1,101,0
2,102,0
3,103,0
4,104,0
5,105,0
6,106,0
7,107,0
8,108,0
9,109,0


In [7]:
yy[yy["label"]==1]

,user,label
15,115,1
47,147,1
77,177,1
82,182,1
97,197,1
...,...,...
1031,1131,1
1050,1150,1
1062,1162,1
1075,1175,1


In [8]:
yy.to_csv('ground_truth.csv', index=False, header=False)

In [9]:
## Example with random scores
 
# data_pred=np.load("subset_training_batch.npz")
data_pred=np.load("/kaggle/input/datasets/victoriaquek/pml-data/subset_training_batch.npz")
X_pred=data_pred["X"]
n_users = len(np.unique(X_pred[:, 0]))

# Generate Random Anomaly Scores
np.random.seed(42)
y_score = np.random.uniform(0, 1, size=n_users)  # random float scores between 0 and 1

print(f"# of predictions:     {len(y_score)}")
print(f"Sample scores:        {y_score[:5]}")

## Save as submission.npz
np.savez('submission.npz', predictions=y_score)
print("\nsubmission.npz saved successfully!")

# of predictions:     220
Sample scores:        [0.37454012 0.95071431 0.73199394 0.59865848 0.15601864]

submission.npz saved successfully!


In [10]:
def prepare_user_features(ratings_df):
    """
    Aggregates transaction-level ratings into user-level feature vectors.
    
    Args:
        ratings_df (pd.DataFrame): Columns ['User', 'Item', 'Rating']
        
    Returns:
        pd.DataFrame: Index is User ID, columns are features.
    """
    # Group by User and calculate statistical features
    user_features = ratings_df.groupby('User').agg(
        rating_count=('Rating', 'count'),
        rating_mean=('Rating', 'mean'),
        rating_std=('Rating', 'std'),
        rating_min=('Rating', 'min'),
        rating_max=('Rating', 'max'),
        unique_items=('Item', 'nunique')
    ).reset_index()
    
    # Fill NaN values in std (occurs if a user has only 1 rating)
    user_features['rating_std'] = user_features['rating_std'].fillna(0)
    
    # Set User as index for easier merging later
    user_features.set_index('User', inplace=True)
    
    return user_features

In [11]:
def train_and_evaluate(ratings_df, labels_df):
    """
    Main pipeline to train Isolation Forest and evaluate using AUC.
    """
    # 1. Feature Engineering
    print("Engineering user features...")
    X_users = prepare_user_features(ratings_df)
    
    # 2. Align Data with Labels
    # Ensure we only keep users that exist in both the ratings and the labels
    common_users = X_users.index.intersection(labels_df['User'])
    
    X_final = X_users.loc[common_users]
    y_final = labels_df.loc[labels_df['User'].isin(common_users)].set_index('User').loc[common_users]['Label']
    
    # Ensure order matches
    y_final = y_final.reindex(X_final.index)
    
    print(f"Total users for analysis: {len(X_final)}")
    print(f"Anomaly rate in dataset: {y_final.mean():.4f}")
    
    # 3. Preprocessing (Scaling)
    # While Tree-based models don't strictly require scaling, 
    # it helps when features have vastly different ranges (e.g., count vs rating 0-5)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_final)
    
    # 4. Initialize Isolation Forest
    # contamination: Expected proportion of outliers. 
    # Since anomalies are "very small proportion", we set this low.
    # Note: For score_samples (used for AUC), contamination does not affect the score values,
    # only the binary predictions.
    model = IsolationForest(
        n_estimators=200, 
        contamination=0.01, # Assumed low anomaly rate
        random_state=42, 
        n_jobs=-1
    )
    
    # 5. Train Model
    print("Training Isolation Forest...")
    model.fit(X_scaled)
    
    # 6. Generate Anomaly Scores
    # model.score_samples returns the "anomaly score" where LOWER = more anomalous.
    # The task requires: HIGHER score = more anomalous.
    # So we invert the sign.
    raw_scores = model.score_samples(X_scaled)
    anomaly_scores = -1 * raw_scores
    
    # 7. Evaluate (AUC)
    auc_score = roc_auc_score(y_final, anomaly_scores)
    
    print(f"Model Evaluation (AUC): {auc_score:.4f}")
    
    # 8. Return Results
    results_df = pd.DataFrame({
        'User': X_final.index,
        'Anomaly_Score': anomaly_scores,
        'True_Label': y_final.values
    })
    
    return results_df, auc_score

In [12]:
np.random.seed(42)
n_normal = 950
n_anomalous = 50
    
# Normal Users: Varied ratings, moderate count
normal_users = np.arange(0, n_normal)
normal_data = []
for u in normal_users:
    n_interactions = np.random.randint(5, 50)
    items = np.random.randint(0, 1000, n_interactions)
    ratings = np.random.choice([1,2,3,4,5], n_interactions, p=[0.1, 0.2, 0.4, 0.2, 0.1])
    for i, r in zip(items, ratings):
        normal_data.append([u, i, r])
            
# Anomalous Users: High count, extreme ratings (e.g., all 5s)
anomalous_users = np.arange(n_normal, n_normal + n_anomalous)
anomalous_data = []
for u in anomalous_users:
    n_interactions = np.random.randint(50, 100) # Higher activity
    items = np.random.randint(0, 1000, n_interactions)
    ratings = np.random.choice([5], n_interactions) # Always 5 stars
    for i, r in zip(items, ratings):
        anomalous_data.append([u, i, r])
            
ratings_df = pd.DataFrame(normal_data + anomalous_data, columns=['User', 'Item', 'Rating'])
labels_df = pd.DataFrame({
    'User': list(normal_users) + list(anomalous_users),
    'Label': [0]*n_normal + [1]*n_anomalous
})

In [13]:
results, auc = train_and_evaluate(ratings_df, labels_df)
print("\nTop 5 Detected Anomalous Users:")
print(results.sort_values(by='Anomaly_Score', ascending=False).head(5))

Engineering user features...
Total users for analysis: 1000
Anomaly rate in dataset: 0.0500
Training Isolation Forest...
Model Evaluation (AUC): 0.9946

Top 5 Detected Anomalous Users:
     User  Anomaly_Score  True_Label
983   983       0.727996           1
959   959       0.719742           1
953   953       0.703612           1
988   988       0.698819           1
978   978       0.694425           1


In [14]:
results.head()

,User,Anomaly_Score,True_Label
0,0,0.404891,0
1,1,0.501627,0
2,2,0.386639,0
3,3,0.389909,0
4,4,0.395901,0


In [18]:
results.sort_values(by='Anomaly_Score', ascending=False).head(30)

,User,Anomaly_Score,True_Label
983,983,0.727996,1
959,959,0.719742,1
953,953,0.703612,1
988,988,0.698819,1
978,978,0.694425,1
961,961,0.694232,1
972,972,0.691246,1
955,955,0.688713,1
996,996,0.688201,1
170,170,0.687998,0
